# Milestone 4.36: Standardizing Column Names and Data Formats

**Objective:** Master the essential skill of standardizing column names and data formats to create clean, predictable, and analysis-ready datasets.

**Key Learning:**
- Messy column names make code harder to write and maintain
- **Standardize names** = Convert to consistent format (lowercase, underscores)
- **Standardize data** = Clean text, ensure consistent formats
- **Apply early** = Standardize at the start of every workflow
- **Be consistent** = Use same rules across all datasets

**Think of it this way:** Standardization is like organizing files in folders with clear naming—everything becomes easier to find and use.

---

## Why Standardization Matters

### The Problem

**Common beginner issues:**
- ❌ Column names with spaces ("First Name" vs "FirstName" vs "first_name")
- ❌ Mixed casing ("Email Address" vs "EMAIL_ADDRESS" vs "email_address")
- ❌ Special characters in names ("Salary ($)"  vs "Price (USD)")
- ❌ Inconsistent text formats ("Active" vs "active" vs "ACTIVE")
- ❌ Mixed date formats ("01/15/2020" vs "2020-01-15" vs "15-Jan-2020")

**Result:** Code becomes fragile, error-prone, and hard to maintain.

### The Solution

**Standardization workflow:**

1. **Standardize Column Names**
   - Convert to lowercase
   - Replace spaces with underscores
   - Remove special characters
   - Apply snake_case convention

2. **Standardize Text Data**
   - Strip extra whitespace
   - Convert to consistent casing
   - Normalize category values

3. **Standardize Formats**
   - Ensure numeric columns are truly numeric
   - Recognize date format issues
   - Handle currency symbols

**Professional habit:** Always standardize at the start of your analysis workflow!

In [ ]:
# Import libraries
import pandas as pd
import numpy as np
import re

print("Pandas version:", pd.__version__)
print("NumPy version:", np.__version__)

---

## Part 1: Understanding the Mess

### Loading Data with Inconsistent Names and Formats

In [ ]:
# Load employee data with messy column names
employees_df = pd.read_csv('../data/raw/employee_records.csv')

print("Employee Data (ORIGINAL):")
print(employees_df.head())
print(f"\nShape: {employees_df.shape}")
print(f"\nColumn names:")
print(employees_df.columns.tolist())

**Observation:** Column names are a mess!
- `"Employee ID"` - Has space
- `"First Name"` - Has space
- `"LAST_NAME"` - All caps with underscore
- `"Email Address"` - Has space
- `"Department Name"` - Has space
- `"Salary ($)"` - Has space AND special characters
- `"Hire Date"` - Has space
- `"Status"` - Only clean one!

**Problem:** Accessing columns requires quotes and exact casing:
```python
employees_df["First Name"]  # Must use quotes
employees_df["Salary ($)"]  # Special characters make it worse
```

### Inspecting Data Format Issues

In [ ]:
# Check data types
print("Data types:")
print(employees_df.dtypes)

print("\nSample values - notice inconsistencies:")
print("\nDepartment Name (mixed casing, extra spaces):")
print(employees_df['Department Name'].unique())

print("\nStatus (mixed casing):")
print(employees_df['Status'].unique())

print("\nFirst Name (extra whitespace):")
print(employees_df['First Name'].tolist())

**Issues found:**
1. **Column names:** Mixed casing, spaces, special characters
2. **Text data:** Extra whitespace, inconsistent casing
3. **Salary:** Stored as object (string) due to "$" symbol
4. **Hire Date:** Mixed date formats ("01/15/2020" vs "2020-01-15")

**These issues prevent:**
- Clean column access (`df.column_name`)
- Proper numeric operations on salary
- Date-based filtering and sorting
- Accurate grouping by department/status

---

## Part 2: Standardizing Column Names

### Method 1: Manual Renaming (Not Scalable)

In [ ]:
# Manual approach - rename specific columns
employees_manual = employees_df.copy()
employees_manual = employees_manual.rename(columns={
    'Employee ID': 'employee_id',
    'First Name': 'first_name',
    'LAST_NAME': 'last_name',
    'Email Address': 'email_address',
    'Department Name': 'department_name',
    'Salary ($)': 'salary',
    'Hire Date': 'hire_date',
    'Status': 'status'
})

print("After manual renaming:")
print(employees_manual.columns.tolist())

**✅ Pro:** Full control over each name

**❌ Con:** Tedious, error-prone, doesn't scale (imagine 50 columns!)

**When to use:** When you need specific custom names for a few columns

### Method 2: Automated Standardization (RECOMMENDED)

In [ ]:
# Function to standardize column names automatically
def standardize_column_names(df):
    """
    Standardize column names to snake_case:
    - Convert to lowercase
    - Replace spaces with underscores
    - Remove special characters
    - Remove extra underscores
    """
    df_clean = df.copy()
    
    # Get current columns
    old_columns = df_clean.columns.tolist()
    
    # Standardize each column name
    new_columns = []
    for col in old_columns:
        # Convert to lowercase
        clean_col = col.lower()
        
        # Replace spaces with underscores
        clean_col = clean_col.replace(' ', '_')
        
        # Remove special characters (keep only letters, numbers, underscores)
        clean_col = re.sub(r'[^a-z0-9_]', '', clean_col)
        
        # Remove multiple consecutive underscores
        clean_col = re.sub(r'_+', '_', clean_col)
        
        # Remove leading/trailing underscores
        clean_col = clean_col.strip('_')
        
        new_columns.append(clean_col)
    
    # Apply new column names
    df_clean.columns = new_columns
    
    # Show changes
    print("Column name changes:")
    for old, new in zip(old_columns, new_columns):
        if old != new:
            print(f"  '{old}' → '{new}'")
    
    return df_clean

# Apply standardization
employees_clean = standardize_column_names(employees_df)

print("\nStandardized columns:")
print(employees_clean.columns.tolist())

**✅ Benefits:**
- Works on any DataFrame automatically
- Consistent results every time
- Scalable to hundreds of columns
- Reusable function

**Result:** All columns now follow `snake_case` convention!

### Method 3: Using Pandas String Methods (Quick Alternative)

In [ ]:
# Quick one-liner approach
employees_quick = employees_df.copy()

# Convert to lowercase and replace spaces
employees_quick.columns = (employees_quick.columns
                           .str.lower()
                           .str.replace(' ', '_', regex=False)
                           .str.replace(r'[^a-z0-9_]', '', regex=True)
                           .str.replace(r'_+', '_', regex=True)
                           .str.strip('_'))

print("Quick method result:")
print(employees_quick.columns.tolist())

**✅ Pro:** Concise, built-in Pandas functionality

**When to use:** Quick scripts, one-off cleaning

### Comparing: Before and After

In [ ]:
# Side-by-side comparison
print("=" * 70)
print("COLUMN NAME COMPARISON")
print("=" * 70)
print(f"{'BEFORE':<35} | {'AFTER':<30}")
print("-" * 70)

for old, new in zip(employees_df.columns, employees_clean.columns):
    print(f"{old:<35} | {new:<30}")

print("=" * 70)

### Benefits of Clean Column Names

In [ ]:
# Before: Must use quotes and exact casing
print("❌ BEFORE (messy):")
print('employees_df["First Name"]  # Requires quotes')
print('employees_df["Salary ($)"]  # Special characters are awkward')

# After: Clean dot notation or simple bracket notation
print("\n✅ AFTER (clean):")
print('employees_clean.first_name  # Clean dot notation')
print('employees_clean.salary  # No special characters')
print('employees_clean["first_name"]  # Or brackets')

# Demonstrate it works
print("\nActually accessing data:")
print(employees_clean.first_name.head(3))

---

## Part 3: Standardizing Text Data

### Problem: Inconsistent Text Formatting

In [ ]:
# Show text inconsistencies
print("Department Names (BEFORE standardization):")
print(employees_clean.department_name.unique())
print(f"Unique values: {employees_clean.department_name.nunique()}")

print("\nStatus (BEFORE standardization):")
print(employees_clean.status.unique())
print(f"Unique values: {employees_clean.status.nunique()}")

**Problem:** Same value appears multiple times due to:
- Mixed casing: "Sales" vs "sales" vs "SALES"
- Extra whitespace: "  Sales  " vs "Sales"

**Impact:**
- Groupby operations count "Sales" and "sales" as different
- Filtering misses records due to casing/spaces
- Statistical summaries are wrong

### Step 1: Strip Whitespace

In [ ]:
# Strip leading and trailing whitespace from text columns
text_columns = ['first_name', 'last_name', 'email_address', 'department_name', 'status']

print("Stripping whitespace from text columns...")
for col in text_columns:
    employees_clean[col] = employees_clean[col].str.strip()
    print(f"  ✓ {col}")

print("\nAfter stripping whitespace:")
print(employees_clean[['first_name', 'department_name']].head())

### Step 2: Standardize Casing

In [ ]:
# Convert to title case (First Letter Capitalized)
employees_clean['first_name'] = employees_clean['first_name'].str.title()
employees_clean['last_name'] = employees_clean['last_name'].str.title()

# Convert to title case for department
employees_clean['department_name'] = employees_clean['department_name'].str.title()

# Convert to title case for status
employees_clean['status'] = employees_clean['status'].str.title()

print("After standardizing casing:")
print("\nDepartment Names:")
print(employees_clean.department_name.unique())
print(f"Unique values: {employees_clean.department_name.nunique()}")

print("\nStatus:")
print(employees_clean.status.unique())
print(f"Unique values: {employees_clean.status.nunique()}")

**Much better!** Now we have:
- "Sales" (not "Sales", "sales", "SALES", "  Sales  ")
- "Active" (not "Active", "active", "ACTIVE")
- "Inactive" (not "inactive")

**Result:** Grouping and filtering now work correctly!

### Alternative: Lowercase for Categories

In [ ]:
# Alternative: Convert categories to lowercase
# (Common for machine learning or standardized lookups)
employees_lower = employees_clean.copy()
employees_lower['department_name'] = employees_lower['department_name'].str.lower()
employees_lower['status'] = employees_lower['status'].str.lower()

print("Lowercase categories:")
print("Department:", employees_lower.department_name.unique())
print("Status:", employees_lower.status.unique())

**Choice:**
- **Title case** - Better for display/reports ("Sales", "Marketing")
- **Lowercase** - Better for matching/ML ("sales", "marketing")

**Be consistent across your project!**

### Verification: Check Distribution

In [ ]:
# Count by department (BEFORE vs AFTER)
print("Department counts (AFTER standardization):")
print(employees_clean.department_name.value_counts())

print("\nStatus counts (AFTER standardization):")
print(employees_clean.status.value_counts())

**✅ Now grouping works correctly!**

Before standardization, "sales" and "Sales" would be counted separately.

---

## Part 4: Standardizing Numeric Formats

### Problem: Salary Column is Text (Object)

In [ ]:
# Check salary data type
print("Salary column:")
print(f"Data type: {employees_clean.salary.dtype}")
print(f"\nSample values:")
print(employees_clean.salary.head())

# Try to calculate mean (will fail!)
try:
    mean_salary = employees_clean.salary.mean()
    print(f"Mean salary: {mean_salary}")
except Exception as e:
    print(f"\n❌ ERROR: {e}")
    print("Cannot calculate mean because salary is stored as text!")

**Problem:** Salary is stored as object (string) due to currency symbols.

**Cannot perform:**
- Mathematical operations (sum, mean, median)
- Comparisons (salary > 80000)
- Sorting by numeric value

### Solution: Clean and Convert to Numeric

In [ ]:
# Remove non-numeric characters and convert to numeric
employees_clean['salary'] = (employees_clean['salary']
                             .str.replace('$', '', regex=False)
                             .str.replace(',', '', regex=False)
                             .str.strip()
                             .astype(float))

print("After standardization:")
print(f"Data type: {employees_clean.salary.dtype}")
print(f"\nSample values:")
print(employees_clean.salary.head())

# Now operations work!
print(f"\n✅ Mean salary: ${employees_clean.salary.mean():,.2f}")
print(f"✅ Max salary: ${employees_clean.salary.max():,.2f}")
print(f"✅ Min salary: ${employees_clean.salary.min():,.2f}")

**Success!** Salary is now truly numeric.

---

## Part 5: Standardizing Date Formats

### Problem: Mixed Date Formats

In [ ]:
# Check hire_date column
print("Hire Date column:")
print(f"Data type: {employees_clean.hire_date.dtype}")
print(f"\nSample values (notice different formats):")
print(employees_clean.hire_date)

**Problem:** Mixed date formats:
- "01/15/2020" (MM/DD/YYYY)
- "2021-05-10" (YYYY-MM-DD)
- "03-22-2019" (MM-DD-YYYY)

**Cannot perform:**
- Date-based filtering
- Calculating tenure
- Sorting by date

### Solution: Convert to Datetime

In [ ]:
# Convert to proper datetime (Pandas will try to infer format)
employees_clean['hire_date'] = pd.to_datetime(employees_clean['hire_date'], 
                                               infer_datetime_format=True)

print("After conversion to datetime:")
print(f"Data type: {employees_clean.hire_date.dtype}")
print(f"\nSample values:")
print(employees_clean.hire_date)

# Now date operations work!
print("\n✅ Date operations now work:")
print(f"Earliest hire: {employees_clean.hire_date.min()}")
print(f"Latest hire: {employees_clean.hire_date.max()}")

**Success!** All dates are now standardized to datetime format.

**Benefits:**
- Can sort chronologically
- Can filter by date range
- Can extract year, month, day
- Can calculate time differences

---

## Part 6: Complete Standardization Workflow

### All-in-One Function

In [ ]:
def standardize_dataframe(df, text_cols=None, numeric_cols=None, date_cols=None):
    """
    Complete standardization workflow for a DataFrame
    
    Parameters:
    -----------
    df : DataFrame
        Input DataFrame
    text_cols : list
        Text columns to standardize (strip, title case)
    numeric_cols : list
        Numeric columns to clean (remove symbols)
    date_cols : list
        Date columns to convert
    
    Returns:
    --------
    df_clean : DataFrame
        Standardized DataFrame
    """
    print("=" * 70)
    print("STANDARDIZATION WORKFLOW")
    print("=" * 70)
    
    df_clean = df.copy()
    
    # Step 1: Standardize column names
    print("\n1. Standardizing column names...")
    old_columns = df_clean.columns.tolist()
    df_clean.columns = (df_clean.columns
                        .str.lower()
                        .str.replace(' ', '_', regex=False)
                        .str.replace(r'[^a-z0-9_]', '', regex=True)
                        .str.replace(r'_+', '_', regex=True)
                        .str.strip('_'))
    print(f"   ✓ Cleaned {len(df_clean.columns)} column names")
    
    # Step 2: Standardize text columns
    if text_cols:
        print("\n2. Standardizing text columns...")
        for col in text_cols:
            if col in df_clean.columns:
                df_clean[col] = df_clean[col].str.strip().str.title()
                print(f"   ✓ {col}")
    
    # Step 3: Standardize numeric columns
    if numeric_cols:
        print("\n3. Standardizing numeric columns...")
        for col in numeric_cols:
            if col in df_clean.columns:
                df_clean[col] = (df_clean[col]
                                .str.replace(r'[$,]', '', regex=True)
                                .str.strip()
                                .astype(float))
                print(f"   ✓ {col}")
    
    # Step 4: Standardize date columns
    if date_cols:
        print("\n4. Standardizing date columns...")
        for col in date_cols:
            if col in df_clean.columns:
                df_clean[col] = pd.to_datetime(df_clean[col], format='mixed')
                print(f"   ✓ {col}")
    
    print("\n" + "=" * 70)
    print("STANDARDIZATION COMPLETE!")
    print("=" * 70)
    
    return df_clean

# Test on employee data
employees_final = standardize_dataframe(
    employees_df,
    text_cols=['first_name', 'last_name', 'email_address', 'department_name', 'status'],
    numeric_cols=['salary'],
    date_cols=['hire_date']
)

print("\nFinal result:")
print(employees_final.head())
print("\nData types:")
print(employees_final.dtypes)

---

## Part 7: Real-World Example - Product Inventory

### Loading Messy Product Data

In [ ]:
# Load product inventory with messy formats
products_df = pd.read_csv('../data/raw/product_inventory.csv')

print("Product Inventory (ORIGINAL):")
print(products_df)
print(f"\nColumns: {products_df.columns.tolist()}")
print(f"\nData types:")
print(products_df.dtypes)

**Issues:**
- Column names: "Product-ID", "Product Name", "Price (USD)", "SKU #"
- Product names: Extra spaces
- Category: Mixed casing ("Electronics", "electronics", "ELECTRONICS")
- Price: Stored as text with "$" symbol
- Availability: Inconsistent ("In Stock", "in stock", "IN STOCK")

### Applying Complete Standardization

In [ ]:
# Apply standardization workflow
products_clean = standardize_dataframe(
    products_df,
    text_cols=['product_name', 'category', 'sku', 'availability'],
    numeric_cols=['price_usd', 'stock_count'],
    date_cols=['last_updated']
)

print("\nCleaned Product Data:")
print(products_clean)
print(f"\nData types:")
print(products_clean.dtypes)

### Verification: Check Unique Values

In [ ]:
# Check category standardization
print("Category (AFTER standardization):")
print(products_clean.category.value_counts())

print("\nAvailability (AFTER standardization):")
print(products_clean.availability.value_counts())

# Calculate statistics
print("\n📊 Statistics (now possible!):")
print(f"Average price: ${products_clean.price_usd.mean():.2f}")
print(f"Total inventory: {products_clean.stock_count.sum()} units")
print(f"Most expensive: ${products_clean.price_usd.max():.2f}")
print(f"Cheapest: ${products_clean.price_usd.min():.2f}")

**✅ Success!** Now we can:
- Group by category correctly
- Calculate price statistics
- Filter by availability
- Sort by any column

---

## Part 8: Before and After Comparison

### Visual Comparison

In [ ]:
# Compare original vs cleaned
print("=" * 90)
print("BEFORE vs AFTER COMPARISON")
print("=" * 90)

print("\n📋 Column Names:")
print(f"{'BEFORE':<45} | {'AFTER':<40}")
print("-" * 90)
for old, new in zip(employees_df.columns, employees_final.columns):
    print(f"{old:<45} | {new:<40}")

print("\n📊 Data Types:")
print(f"{'Column':<30} | {'BEFORE':<20} | {'AFTER':<20}")
print("-" * 90)
for col_old, col_new in zip(employees_df.columns, employees_final.columns):
    print(f"{col_new:<30} | {str(employees_df[col_old].dtype):<20} | {str(employees_final[col_new].dtype):<20}")

print("\n📈 Sample Data:")
print("\nBEFORE:")
print(employees_df[['First Name', 'Department Name', 'Salary ($)', 'Status']].head(3))
print("\nAFTER:")
print(employees_final[['first_name', 'department_name', 'salary', 'status']].head(3))

print("\n" + "=" * 90)

---

## Summary: Key Takeaways

### Standardization Checklist

**Column Names:**
- ✅ Convert to lowercase
- ✅ Replace spaces with underscores
- ✅ Remove special characters
- ✅ Use snake_case convention
- ✅ Keep names descriptive but concise

**Text Data:**
- ✅ Strip leading/trailing whitespace
- ✅ Standardize casing (title case or lowercase)
- ✅ Normalize category values
- ✅ Ensure consistency within columns

**Numeric Data:**
- ✅ Remove currency symbols ($, €, etc.)
- ✅ Remove thousand separators (commas)
- ✅ Convert to proper numeric type (int/float)
- ✅ Verify operations work (mean, sum, etc.)

**Date Data:**
- ✅ Convert to datetime type
- ✅ Handle mixed formats
- ✅ Enable date-based operations

### Code Template

```python
# Complete standardization workflow
def standardize_dataframe(df, text_cols=None, numeric_cols=None, date_cols=None):
    df_clean = df.copy()
    
    # 1. Standardize column names
    df_clean.columns = (df_clean.columns
                        .str.lower()
                        .str.replace(' ', '_', regex=False)
                        .str.replace(r'[^a-z0-9_]', '', regex=True))
    
    # 2. Standardize text
    if text_cols:
        for col in text_cols:
            df_clean[col] = df_clean[col].str.strip().str.title()
    
    # 3. Standardize numeric
    if numeric_cols:
        for col in numeric_cols:
            df_clean[col] = (df_clean[col]
                            .str.replace(r'[$,]', '', regex=True)
                            .astype(float))
    
    # 4. Standardize dates
    if date_cols:
        for col in date_cols:
            df_clean[col] = pd.to_datetime(df_clean[col])
    
    return df_clean
```

### Critical Points

✅ **Standardize early** - Do this at the start of every analysis

✅ **Be consistent** - Use same rules across all datasets

✅ **Use snake_case** - Industry standard for column names

✅ **Document changes** - Keep record of transformations

✅ **Verify results** - Check data types and sample values after

✅ **Create reusable functions** - Don't repeat yourself

### Benefits of Standardization

**Code Quality:**
- Clean column access (`df.column_name`)
- Less error-prone code
- Easier to read and maintain

**Data Quality:**
- Correct groupby operations
- Accurate filtering
- Valid statistical calculations

**Workflow Efficiency:**
- Easier to merge datasets
- Reusable code across projects
- Scales to large datasets

**Team Collaboration:**
- Predictable naming conventions
- Consistent formatting
- Easier to share and understand

---

## Next Steps

1. **Apply standardization to all your datasets**
2. **Create your own standardization function**
3. **Use snake_case for all column names**
4. **Always standardize at the start of analysis**
5. **Record your 2-minute demonstration video**

**You're now ready to standardize data like a professional!** 🎉